# HUD USPS ZIP Crosswalk Toolkit — Demo

This notebook walks through a complete example of using the HUD USPS ZIP Crosswalk Toolkit to merge survey data collected at the ZIP code level with county level administrative data.

The example uses synthetic data so the notebook runs end to end without requiring a HUD API token or access to real survey data. The API call section shows the real code you would use with a valid token.

**Three classes are used in this workflow:**
- `DataLoad` — retrieves HUD crosswalk data from the API and loads local files
- `DataProcessing` — profiles and validates ZIP code data quality before merging
- `DataMerge` — maps ZIP codes to county FIPS codes using the HUD crosswalk

---

## 0. Setup

Import the required packages and the toolkit classes.

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../')

from src.hud_crosswalk.dataload import DataLoad
from src.hud_crosswalk.dataprocessing import DataProcessing
from src.hud_crosswalk.datamerge import DataMerge

---

## 1. Retrieving HUD Crosswalk Data from the API

The `DataLoad` class handles authentication and data retrieval from the HUD USPS Crosswalk API.
A free API token is required — register at https://www.huduser.gov/portal/dataset/uspszip-api.html

The API token is passed explicitly to `api_call()` rather than stored as an instance attribute.
This avoids silent failures when working with multiple datasets or workflows.

The code below shows how to retrieve ZIP to county crosswalk data (Type 2) for all ZIP codes
in the United States for Q3 2022. **This cell is commented out for the demo — uncomment and run with a valid token.**

In [ ]:
# dl = DataLoad()
#
# params = dl.huds_url_modifier(
#     crosswalk_type=2,            # Type 2 = ZIP to County
#     query='All',                 # All ZIP codes
#     year=2022,
#     quarter=3
# )
#
# huds_df = dl.api_call('your_api_token', params)    # token passed explicitly
# huds_df.head()

---

## 2. Synthetic Data

For this demo we create two synthetic datasets:

- **Survey data** — respondent ZIP codes from Virginia, mimicking a survey collected at the ZIP code level
- **HUD crosswalk data** — ZIP to county mapping with residential address ratios, mimicking the HUD API response

Some ZIP codes in the survey data intentionally span multiple counties to demonstrate how the toolkit resolves this.
A few rows also contain intentional data quality issues to demonstrate the profiling step.

In [ ]:
# Synthetic survey data — Virginia ZIP codes
# Includes two intentional data quality issues:
# - '2203A' contains a non-numeric character
# - '2203' is only 4 digits (too short)
survey_data = pd.DataFrame({
    'respondent_id': range(1, 11),
    'zip': ['22031', '2203A', '22033', '20110', '20111',
            '23451', '23452', '22901', '2203', '20176'],
    'response': np.random.randint(1, 6, 10)
})

print("Survey Data:")
print(survey_data)

In [ ]:
# Synthetic HUD crosswalk data
# Some ZIP codes appear multiple times — they span more than one county
huds_data = pd.DataFrame({
    'zip':       ['22031', '22032', '22033', '22033', '20110',
                  '20111', '20111', '23451', '23452', '22901',
                  '22901', '22902', '20176', '20176'],
    'geoid':     ['51059', '51059', '51059', '51600', '51683',
                  '51683', '51610', '51650', '51650', '51003',
                  '51540', '51003', '51107', '51610'],
    'res_ratio': [1.0, 1.0, 0.73, 0.27, 1.0,
                  0.65, 0.35, 1.0, 1.0, 0.80,
                  0.20, 1.0, 0.60, 0.40]
})

print("HUD Crosswalk Data:")
print(huds_data)

---

## 3. Data Profiling — DataProcessing

Before merging, always profile the ZIP code column in the survey data to check for data quality issues.
The `DataProcessing` class provides two methods for this:

- `data_profile()` — counts missing values, non-numeric characters, and incorrect length. If issues are found,
  it automatically calls `data_filter()` and returns both a summary dictionary and a DataFrame of problematic rows.
- `data_filter()` — returns a DataFrame of rows that failed one or more checks, with flag columns indicating which check failed.

Resolving data quality issues before merging prevents silent mismatches where a ZIP code fails to match
any county in the HUD crosswalk.

In [ ]:
dp = DataProcessing()

# Profile the ZIP column
# Returns dict if no issues found
# Returns tuple (dict, DataFrame) if issues found
result = dp.data_profile(survey_data, 'zip')

if isinstance(result, tuple):
    profile, problems = result
    print("\nProblematic Rows:")
    print(problems)
else:
    profile = result
    print("\nNo issues found — safe to proceed with merge.")

In [ ]:
# For this demo, drop the problematic rows before merging
# In practice, researchers should inspect and resolve these manually
# e.g. correct typos, look up the correct ZIP, or document and exclude
clean_survey = survey_data[~survey_data['zip'].isin(problems['zip'])].copy()
print(f"Rows before cleaning: {len(survey_data)}")
print(f"Rows after cleaning:  {len(clean_survey)}")
print(clean_survey)

---

## 4. ZIP to County Merge — DataMerge

With clean data, run the full ZIP to county mapping pipeline using the `DataMerge` class.

The `run()` method handles everything in the correct order:
1. Standardizes ZIP and geoid columns (zero fill, strip whitespace)
2. Builds a ZIP to county mapping dictionary from the HUD data
3. Maps county FIPS codes onto the survey data by ZIP
4. Expands multi-county ZIP codes into separate columns
5. Resolves multi-county ZIP codes to a single county using residential address ratios

By default, `DataMerge` assumes standard HUD column names (`zip`, `geoid`, `res_ratio`).
If your HUD data uses different column names, pass them at initialization — see Section 6.

In [ ]:
dm = DataMerge()

# Run the full pipeline
merged = dm.run(clean_survey, huds_data, df_zip='zip')

print("Merged Data:")
print(merged)

In [ ]:
# Inspect ZIP codes that mapped to multiple counties
# assigned_geoid is the county with the highest residential address ratio
geoid_cols = [col for col in merged.columns if col.startswith('geoid_')]
multi_county = merged[merged['geoid_2'] != 0]

print("ZIP codes spanning multiple counties:")
print(multi_county[['zip'] + geoid_cols + ['assigned_geoid']])

---

## 5. Loading Local Files — DataLoad

If you have already downloaded the HUD crosswalk data and saved it locally,
use `data_load()` to load it directly without calling the API.
Partial filenames are supported — the method matches any file containing the provided string.

In [ ]:
# dl = DataLoad()
#
# Load by partial filename — no need to know the exact name or extension
# huds_df = dl.data_load('/path/to/data/folder', 'huds_county')
# survey_df = dl.data_load('/path/to/data/folder', 'survey_data')

---

## 6. Custom HUD Column Names

If your HUD data uses different column names than the defaults,
pass them at initialization. The defaults are `zip`, `geoid`, and `res_ratio`
based on standard HUD API column naming.

In [ ]:
# Example with custom column names
huds_data_custom = huds_data.rename(columns={
    'zip': 'zipcode',
    'geoid': 'fips',
    'res_ratio': 'ratio'
})

dm_custom = DataMerge(
    huds_zip='zipcode',
    huds_geoid='fips',
    huds_res_ratio='ratio'
)

result_custom = dm_custom.run(clean_survey, huds_data_custom, df_zip='zip')
print(result_custom)

---

## 7. Running Individual Steps

For more control over the pipeline, individual methods can be called separately.
This is useful if you want to inspect intermediate outputs or only need specific steps.
The mapping dictionary is returned explicitly and passed to the next step —
this avoids silent failures when running multiple workflows from the same instance.

In [ ]:
dm = DataMerge()

# Step 1 — build mapping dictionary from HUD data
col_dict = dm.zip_to_location(huds_data)
print("Mapping Dictionary (first 3 entries):")
print(dict(list(col_dict.items())[:3]))

In [ ]:
# Step 2 — assign geoids to survey data
# col_dict is passed explicitly — safe to use with multiple workflows
survey_with_geoids = dm.assign_geoid_main_dataframe(clean_survey, 'zip', col_dict)
print("Survey Data with Geoids:")
print(survey_with_geoids[['zip', 'geoid_huds']])

In [ ]:
# Step 3 — expand multi-county ZIP codes into separate columns
survey_expanded = dm.expand_geoid_columns(survey_with_geoids, 'geoid_huds')
geoid_cols = [col for col in survey_expanded.columns if col.startswith('geoid_')]
print("Expanded Geoid Columns:")
print(survey_expanded[['zip'] + geoid_cols])

In [ ]:
# Step 4 — resolve multi-county ZIPs using residential address ratios
highest_values, nested_dict = dm.resolve_multi_county_zip(huds_data)
survey_expanded['assigned_geoid'] = survey_expanded['zip'].map(highest_values)

print("Final Assignment:")
print(survey_expanded[['zip'] + geoid_cols + ['assigned_geoid']])

---

## 8. Notes

- Always run `DataProcessing.data_profile()` before merging — unresolved ZIP code issues cause silent mismatches
- The HUD crosswalk file is updated quarterly — use the file that corresponds to your data collection period
- Unmatched ZIP codes (common for PO boxes) are returned as `NaN` in the `geoid_huds` column — inspect and document these before dropping
- This toolkit currently supports ZIP to county crosswalk only (Type 2). Support for other crosswalk types is planned for future versions
- For a detailed discussion of the methodology and its limitations see `methodology.md`

## References

Sadler, R. C. (2019). Misalignment between ZIP codes and municipal boundaries: A problem for public health.

Wilson, R., & Din, A. (2018). Understanding and enhancing the U.S. Department of Housing and Urban Development's ZIP code crosswalk files.

HUD USPS Crosswalk API: https://www.huduser.gov/portal/dataset/uspszip-api.html